### Generic SSL LSTM Training

This notebook trains a stacked LSTM classifier for any preprocessed SSL dataset.

It assumes you have already run `preprocess_ssl_generic.ipynb` and have a
`*_sequences.pkl` file under `dataset/processed/<SSL_NAME>/`.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow :", tf.__version__)
print("GPUs       :", tf.config.list_physical_devices("GPU"))

# Dataset configuration
SSL_NAME = "ssl-alphabet-converted"  # change to any ssl-* dataset
BASE_NAME = SSL_NAME.replace("-", "_")

# Adjust if your project lives elsewhere
PROJECT_ROOT = Path("C:/Redmi")

DATA_PATH = (
    PROJECT_ROOT
    / "dataset"
    / "processed"
    / SSL_NAME
    / f"{BASE_NAME}_sequences.pkl"
)
print("Loading from:", DATA_PATH)

In [ ]:
with open(DATA_PATH, "rb") as f:
    data = pickle.load(f)

X_train = data["X_train"]
X_val   = data["X_val"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_val   = data["y_val"]
y_test  = data["y_test"]
le          = data["label_encoder"]
num_classes = data["num_classes"]
SEQ_LEN     = int(data["seq_len"])
STRIDE      = int(data["stride"])

print("X_train :", X_train.shape, " (sequences, timesteps, features)")
print("X_val   :", X_val.shape)
print("X_test  :", X_test.shape)
print("Classes :", num_classes)
print("SEQ_LEN :", SEQ_LEN)
print("STRIDE  :", STRIDE)
print("Labels  :", list(le.classes_))

In [ ]:
n_train, n_steps, n_feats = X_train.shape

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.reshape(-1, n_feats)).reshape(
    n_train, n_steps, n_feats
)
X_val_s   = scaler.transform(X_val.reshape(-1, n_feats)).reshape(X_val.shape)
X_test_s  = scaler.transform(X_test.reshape(-1, n_feats)).reshape(X_test.shape)

print("Scaled X_train shape:", X_train_s.shape)
print("Train mean ≈", f"{X_train_s.mean():.4f}",
      "std ≈", f"{X_train_s.std():.4f}")

In [ ]:
model = keras.Sequential(
    [
        layers.Input(shape=(SEQ_LEN, n_feats)),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(64),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation="softmax"),
    ]
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
unique_classes = np.unique(y_train)
weights = compute_class_weight("balanced", classes=unique_classes, y=y_train)
class_weight = dict(zip(unique_classes.astype(int), weights))

EPOCHS     = 100
BATCH_SIZE = 32

cb_list = [
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=15,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_accuracy",
        mode="max",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    X_train_s,
    y_train,
    validation_data=(X_val_s, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=cb_list,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["loss"], label="Train")
ax1.plot(history.history["val_loss"], label="Val")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(history.history["accuracy"], label="Train")
ax2.plot(history.history["val_accuracy"], label="Val")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
print("Test accuracy :", f"{test_acc:.4f}")
print("Test loss     :", f"{test_loss:.4f}")

y_probs = model.predict(X_test_s, verbose=0)
y_pred = y_probs.argmax(axis=1)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=list(le.classes_)))

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=False,
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_,
)
plt.title(f"Confusion Matrix for {SSL_NAME}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models" / BASE_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path  = MODEL_DIR / f"{BASE_NAME}_lstm.keras"
scaler_path = MODEL_DIR / f"{BASE_NAME}_lstm_scaler.pkl"
meta_path   = MODEL_DIR / f"{BASE_NAME}_sequences.pkl"  # copy of dataset dict

model.save(model_path)
print("Model saved to", model_path)

with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
print("Scaler saved to", scaler_path)

with open(meta_path, "wb") as f:
    pickle.dump(data, f)
print("Metadata saved to", meta_path)